# Feature selection by category: Comorbidity claims

Finds the strongest **comorbidity-claims** features (NBA 3.0 HCP-level claim features: 09 claim comorbidity, 10 Kerendia prior writing, 11 adoption), separately for **existing writers** and **non-writers**, on the 10k-HCP sample with all 28 cohorts. Reads only this category's 3 source tables and fills empty values with the Combine rules.

Same method as every category notebook: the category config (cell 3) is the only difference; all steps run from [00_category_feature_selection_core](./00_category_feature_selection_core) via `%run` (cell 4). **Part 1:** writer definition (`TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH` > 0, empty = 0), writers / non-writers data sets, mean / median analysis (`DISCRIMINATION` = \|2 x AUC - 1\|). **Part 2:** NBA 3.1 clean-up (stages 1-4, flags 5-6), one over-fitted LightGBM Poisson model per group on all 28 cohorts, SHAP, best features. Outputs: `feature_selection_by_category/<RUN_NAME>/comorb_claims/`.

In [0]:
%pip install lightgbm==4.5.0 --quiet
dbutils.library.restartPython()

In [0]:
CATEGORY       = "comorb_claims"
CATEGORY_LABEL = "Comorbidity claims: 09 claim comorbidity, 10 Kerendia prior writing, 11 adoption"
ADD_CALENDAR   = False

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"

def _t(name, sub, path, fmt, cohort_col, drop_cols=()):
    """One source table, defined as in NBA_3_1_Combine_Physician_Data (cell 2)."""
    return dict(name=name, sub_category=sub, path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols))

SOURCE_TABLES = [
    _t("claim_comorbidity", "Comorbidity", DEV + "hcp_features/claim_comorbidity", "delta", "cohort_id"),
    _t("kerendia_prior_writing", "Prior writing", DEV + "hcp_features/kerendia_prior_writing", "delta", "cohort_id"),
    _t("adoption_features", "Adoption", DEV + "hcp_features/adoption_features", "delta", "cohort_id"),
]

In [0]:
%run ./00_category_feature_selection_core